# Supply Chain Service Level Tracking Analysis

In [1]:
# Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Cleaning & Exploration

In [2]:
# Load data
customers = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_customers.csv")
date = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_date.csv")
products = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_products.csv")
targets = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_targets_orders.csv")
order_lines = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/fact_order_lines.csv")


### Preview data set

In [3]:
customers.head(3)

,customer_id,customer_name,city
0,789201,Rel Fresh,Surat
1,789202,Rel Fresh,Ahmedabad
2,789203,Rel Fresh,Vadodara


In [4]:
date.head(3)

,date,mmm_yy,week_no
0,01-Apr-22,01-Apr-22,W 14
1,03-Apr-22,01-Apr-22,W 15
2,04-Apr-22,01-Apr-22,W 15


In [5]:
products.head(3)

,product_id,product_name,category
0,25891101,AM Milk 500,Dairy
1,25891102,AM Milk 250,Dairy
2,25891103,AM Milk 100,Dairy


In [6]:
targets.head(3)

,customer_id,ontime_target%,infull_target%,otif_target%
0,789201,87,81,70
1,789202,85,81,69
2,789203,92,76,70


In [7]:
order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,"Tuesday, March 1, 2022",789203,25891601,110,"Friday, March 4, 2022","Friday, March 4, 2022",110
1,FMR32320302,"Tuesday, March 1, 2022",789320,25891203,347,"Wednesday, March 2, 2022","Wednesday, March 2, 2022",347
2,FMR33320501,"Tuesday, March 1, 2022",789320,25891203,187,"Thursday, March 3, 2022","Thursday, March 3, 2022",150


### Standardise column names for all tables

In [8]:
# Standardise column names
for df in [customers, date, products, targets, order_lines]:
    df.columns = df.columns.str.lower().str.strip()

### Validate **`customers`** table
- data types
- missing values
- duplicates

In [9]:
customers.dtypes

customer_id       int64
customer_name    object
city             object
dtype: object

In [10]:
customers.isnull().sum()

customer_id      0
customer_name    0
city             0
dtype: int64

In [11]:
customers.duplicated().sum()

np.int64(0)

In [12]:
customers.duplicated(subset=['customer_id', 'customer_name', 'city']).sum()

np.int64(0)

### Validate **`date`** table
- data types
- missing values
- duplicates

In [13]:
date.dtypes

date       object
mmm_yy     object
week_no    object
dtype: object

#### parse datetime data type for columns:
- date
- mmm-yy

In [14]:
# Convert date column
date['date'] = pd.to_datetime(date['date'], format='%d-%b-%y')

# Convert mmm_yy to datetime (month-level)
date['mmm_yy'] = pd.to_datetime(date['mmm_yy'], format='%d-%b-%y')

# week_no stays as text
date['week_no'] = date['week_no'].astype('string')

date.dtypes

date       datetime64[ns]
mmm_yy     datetime64[ns]
week_no    string[python]
dtype: object

In [15]:
date.head(3)

,date,mmm_yy,week_no
0,2022-04-01,2022-04-01,W 14
1,2022-04-03,2022-04-01,W 15
2,2022-04-04,2022-04-01,W 15


In [16]:
date.isnull().sum()

date       0
mmm_yy     0
week_no    0
dtype: int64

In [17]:
date.duplicated().sum()

np.int64(0)

### Validate **`products`** table
- data types
- missing values
- duplicates

In [18]:
products.dtypes

product_id       int64
product_name    object
category        object
dtype: object

In [19]:
products.isnull().sum()

product_id      0
product_name    0
category        0
dtype: int64

In [20]:
products.duplicated(subset=['product_id', 'product_name']).sum()

np.int64(0)

### Validate **`targets`** table
- data types
- missing values
- duplicates

In [21]:
targets.dtypes

customer_id       int64
ontime_target%    int64
infull_target%    int64
otif_target%      int64
dtype: object

In [22]:
targets.isnull().sum()

customer_id       0
ontime_target%    0
infull_target%    0
otif_target%      0
dtype: int64

In [23]:
targets.duplicated(subset=['customer_id']).sum()

np.int64(0)

In [24]:
# Adjust column names
targets = targets.rename(columns={
    'ontime_target%': 'ontime_target_pct',
    'infull_target%': 'infull_target_pct',
    'otif_target%': 'otif_target_pct'
})

In [25]:
targets.columns.to_list()

['customer_id', 'ontime_target_pct', 'infull_target_pct', 'otif_target_pct']

### Validate **`order_lines`** table
- data types
- missing values
- duplicates

In [26]:
order_lines.dtypes

order_id                object
order_placement_date    object
customer_id              int64
product_id               int64
order_qty                int64
agreed_delivery_date    object
actual_delivery_date    object
delivery_qty             int64
dtype: object

#### parse datetime data type for columns:
- order_placement_date
- agreed_delivery_date
- actual_delivery_date

In [27]:
order_lines['order_placement_date'] = pd.to_datetime(order_lines['order_placement_date'])
order_lines['agreed_delivery_date'] = pd.to_datetime(order_lines['agreed_delivery_date'])
order_lines['actual_delivery_date'] = pd.to_datetime(order_lines['actual_delivery_date'])

order_lines.dtypes

order_id                        object
order_placement_date    datetime64[ns]
customer_id                      int64
product_id                       int64
order_qty                        int64
agreed_delivery_date    datetime64[ns]
actual_delivery_date    datetime64[ns]
delivery_qty                     int64
dtype: object

In [28]:
order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150


In [29]:
order_lines.isnull().sum()

order_id                0
order_placement_date    0
customer_id             0
product_id              0
order_qty               0
agreed_delivery_date    0
actual_delivery_date    0
delivery_qty            0
dtype: int64

In [30]:
order_lines.duplicated().sum()

np.int64(0)

### Merge  **`customers`** & **`products`** with **`orders_lines`**

In [31]:
order_lines = (order_lines.merge(customers, on='customer_id', how='left').merge(products, on='product_id', how='left'))

order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy


In [32]:
order_lines.shape

(57096, 12)

### Create Core Service Level Agreement (SLA) Flags
- On-Time Delivery
- In-Full Delivery
- OTIF (On-Time + In-Full)

In [33]:
# On-Time Delivery
order_lines['on_time_flag'] = (
    order_lines['actual_delivery_date'] 
    <= order_lines['agreed_delivery_date']
)

# In-Full Delivery
order_lines['in_full_flag'] = (
    order_lines['delivery_qty'] 
    >= order_lines['order_qty']
)

# OTIF (On-Time + In-Full)
order_lines['otif_flag'] = (
    order_lines['on_time_flag'] & order_lines['in_full_flag']
)

order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category,on_time_flag,in_full_flag,otif_flag
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages,True,True,True
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy,True,True,True
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy,True,False,False


### Fill Rate Metrics

#### 1. Line Fill Rate (LiFR)

**Definition**: Number of order lines fulfilled in full / total order lines

So at line level, we first create a **`line_fulfilled_flag`**:

**LiFR** is calculated later during aggregation as: **`SUM(line_fulfilled_flag)` / `COUNT(order_lines)`**

In [34]:
# line_fulfilled_flag - boolean (true/false)
order_lines['line_fulfilled_flag'] = (order_lines['delivery_qty'] >= order_lines['order_qty'])

# Convert to numeric (needed for aggregation later)
order_lines['line_fulfilled_flag'] = (order_lines['line_fulfilled_flag'].astype(int))


#### 2. Volume Fill Rate (VoFR)

At line level:

**Definition**: Total quantity delivered / total quantity ordered

Final **VoFR** is calculated later as: **`SUM(delivery_qty)` / `SUM(order_qty)`**


In [35]:
# VoFR (line level)
order_lines['volume_fill_rate'] = (order_lines['delivery_qty'] / order_lines['order_qty']).round(2)

### Delay Calculation
- `delay_days`
- `late_delivery_flag`

In [36]:
# delay_days
order_lines['delay_days'] = (order_lines['actual_delivery_date'] - order_lines['agreed_delivery_date']).dt.days

# late_delivery_flag
order_lines['late_delivery_flag'] = order_lines['delay_days'] > 0

### Date Enrichment:

Extract time features for drilling

In [37]:
order_lines['year'] = order_lines['order_placement_date'].dt.year
order_lines['month'] = order_lines['order_placement_date'].dt.month
order_lines['week'] = order_lines['order_placement_date'].dt.isocalendar().week
order_lines['day'] = order_lines['order_placement_date'].dt.date

### Target Mapping

Join **`targets`** to **`order_lines`**

In [38]:
order_lines = order_lines.merge(targets, on='customer_id', how='left')

### Compute:
- On-Time vs Target
- In-Full vs Target
- OTIF vs Target

In [39]:
order_lines['ontime_vs_target'] = (order_lines['on_time_flag']*1 - order_lines['ontime_target_pct']/100)

In [40]:
order_lines['infull_vs_target'] = (order_lines['in_full_flag'].astype(int) - order_lines['infull_target_pct']/100)

In [41]:
order_lines['otif_vs_target'] = (order_lines['otif_flag'].astype(int) - order_lines['otif_target_pct']/100)

In [42]:
order_lines.head(5)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,...,year,month,week,day,ontime_target_pct,infull_target_pct,otif_target_pct,ontime_vs_target,infull_vs_target,otif_vs_target
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,...,2022,3,9,2022-03-01,92,76,70,0.08,0.24,0.30
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,...,2022,3,9,2022-03-01,91,81,74,0.09,0.19,0.26
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,...,2022,3,9,2022-03-01,91,81,74,0.09,-0.81,-0.74
3,FMR34220601,2022-03-01,789220,25891203,235,2022-03-04,2022-03-04,235,Atlas Stores,Surat,...,2022,3,9,2022-03-01,91,76,69,0.09,0.24,0.31
4,FMR33703603,2022-03-01,789703,25891203,176,2022-03-03,2022-03-03,176,Sorefoz Mart,Vadodara,...,2022,3,9,2022-03-01,85,78,66,0.15,0.22,0.34


In [43]:
order_lines.columns.to_list()

['order_id',
 'order_placement_date',
 'customer_id',
 'product_id',
 'order_qty',
 'agreed_delivery_date',
 'actual_delivery_date',
 'delivery_qty',
 'customer_name',
 'city',
 'product_name',
 'category',
 'on_time_flag',
 'in_full_flag',
 'otif_flag',
 'line_fulfilled_flag',
 'volume_fill_rate',
 'delay_days',
 'late_delivery_flag',
 'year',
 'month',
 'week',
 'day',
 'ontime_target_pct',
 'infull_target_pct',
 'otif_target_pct',
 'ontime_vs_target',
 'infull_vs_target',
 'otif_vs_target']

In [44]:
order_lines[['on_time_flag', 'in_full_flag', 'otif_flag', 'ontime_target_pct', 'infull_target_pct', 'otif_target_pct', 'ontime_vs_target',
             'infull_vs_target',
             'otif_vs_target']].head()

,on_time_flag,in_full_flag,otif_flag,ontime_target_pct,infull_target_pct,otif_target_pct,ontime_vs_target,infull_vs_target,otif_vs_target
0,True,True,True,92,76,70,0.08,0.24,0.30
1,True,True,True,91,81,74,0.09,0.19,0.26
2,True,False,False,91,81,74,0.09,-0.81,-0.74
3,True,True,True,91,76,69,0.09,0.24,0.31
4,True,True,True,85,78,66,0.15,0.22,0.34
